# DNG vs CC TIFF Investigation & CNN Comparison (v3 — paired, repeated CV, controlled seeding, paired significance test)

This notebook investigates differences between:
- **DNG source images** from `F:\Aymen\server\...\2026-02-26_Scorings_XX`
- **CC pipeline TIFF outputs** from `P:\2026_Aymen\server\...\OUTPUT\nocc\CSVS01`

**What changed vs. v2 of this notebook:**
1. CNN training uses **only the matched DNG↔TIFF pairs** (213 images) for both sources, instead
   of the full independent 224 DNG / 213 TIFF lists — DNG and TIFF are trained and evaluated on
   the exact same set of physical items.
2. `matched_pairs` is now **sorted by `base_name` before any K-fold split**. This matters:
   `StratifiedKFold`/`RepeatedStratifiedKFold` with `shuffle=True` produce *different* fold
   membership for the *same* `random_state` if the input row order differs — fixing the order
   makes this notebook's folds reproducible run-to-run and comparable against any other notebook
   that also sorts by `base_name` first.
3. Upgraded from a single **5-fold** CV to a **repeated 5-fold × 3-repeat CV** (`K_FOLDS ×
   N_REPEATS` = 15 independent splits), with the identical split sequence reused for both DNG and
   TIFF. This cuts the noise in the final comparison roughly 3x relative to a single 5-fold pass.
4. **Controlled seeding**: `reset_seed(seed)` is called identically right before building the TIFF
   model and right before building the DNG model within each split, so both start from the same
   initial weights and see the same augmentation RNG stream — only the image source differs.
5. Final metrics are computed from **pooled, repeat-averaged out-of-fold (OOF) predictions**: every
   matched pair is held out once per repeat (in a different fold each time), and its final
   prediction is the average of those `N_REPEATS` independent held-out predictions — lower
   variance than a single OOF pass, still fully leak-free.
6. Added a **paired significance test** (paired t-test + Wilcoxon signed-rank) on the per-split
   (DNG − TIFF) differences, since DNG and TIFF always share identical split membership — a more
   rigorous answer to "is this gap real?" than a flat 0.05 threshold on the pooled numbers alone.

Everything else (DNG/TIFF decoding, matching logic, model architecture, transforms) is unchanged
from v2 — those parts were already correct.

## Contents
1. Imports & Configuration
2. Discover and Match Files
3. Visual Comparison
4. Statistical Comparison (pixel distributions, ΔE)
5. CNN Model Definition
6. Custom Datasets & Transforms
7. Prepare Paired Datasets for CNN Training
8. Repeated Stratified K-Fold Cross-Validation Setup
9. Cross-Validated Training (paired, controlled seeding)
10. Per-Split Results (mean ± std)
10b. Paired Significance Test (DNG − TIFF)
11. Pooled, Repeat-Averaged Out-of-Fold Metrics Comparison
12. Per-Class Statistical Summary (OOF)
13. Confusion Matrix (OOF)
14. Predicted vs True Scatter Plots (OOF)
15. Frieze of Predictions (OOF)
16. Final Comparison Summary


## 1. Imports & Configuration

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from glob import glob
import os
import random
from tqdm.auto import tqdm
from skimage import io
from skimage.transform import resize
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from scipy.stats import ttest_rel, wilcoxon
import rawpy
import colour
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

mpl.rcParams['figure.figsize'] = 16, 8
plt.rcParams['font.size'] = 10


def reset_seed(seed):
    """Controlled seeding: call this identically before building/training each model so
    DNG and TIFF runs (within the same fold) start from the same initial weights and see
    the same augmentation RNG stream."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


reset_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Libraries imported successfully!")


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Source DNG dataset path
DNG_BASE_PATH = r"F:\Aymen\server\Database_CSVS\CSVS_Zucchini_202602_LaMojonena\Zucchini_COLOR_Evaluation\Zucchini_GREEN_FOR_SPAIN_on_LVS_platform"
DNG_FOLDER_PATTERN = "2026-02-26_Scorings_"  # + 01, 02, ... 09

# CC pipeline TIFF output path
TIFF_BASE_PATH = r"P:\2026_Aymen\server\Database_CSVS\CSVS_Zucchini_202602_LaMojonena\Zucchini_COLOR_Evaluation\Zucchini_GREEN_FOR_SPAIN_on_LVS_platform\OUTPUT\nocc\CSVS01"

# Score range
SCORE_RANGE = list(range(1, 9))  # 1-8

# CNN Training parameters
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
NUM_EPOCHS = 400
LEARNING_RATE = 0.001
EARLY_STOPPING_PATIENCE = 30
LR_PATIENCE = 8
LR_FACTOR = 0.5
DEPTH = 16
K_FOLDS = 5   # stratified cross-validation, same folds for DNG and TIFF
N_REPEATS = 3   # repeated CV: K_FOLDS x N_REPEATS independent splits for a more robust estimate
SEED = 42

print("Configuration:")
print(f"  DNG path: {DNG_BASE_PATH}")
print(f"  TIFF path: {TIFF_BASE_PATH}")
print(f"  Image size: {IMG_SIZE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Early stopping patience: {EARLY_STOPPING_PATIENCE}")
print(f"  LR patience: {LR_PATIENCE}")
print(f"  K-fold splits: {K_FOLDS}  |  Repeats: {N_REPEATS}  |  Total splits: {K_FOLDS * N_REPEATS}")


## 2. Discover and Match Files

- DNG files are in folders named `2026-02-26_Scorings_01`, `_02`, etc.
- TIFF files are in score subfolders like `Score=01/` inside CSVS01
- Matching is done by base filename (extension differs)
- **Unchanged from the original notebook** — this logic was already correct (0 score mismatches).


In [ ]:
def discover_dng_files(base_path, folder_pattern, score_range):
    dng_dict = {}
    total = 0
    for score in score_range:
        folder_name = f"{folder_pattern}{score:02d}"
        folder_path = os.path.join(base_path, folder_name)
        if not os.path.exists(folder_path):
            print(f"  [SKIP] Folder not found: {folder_name}")
            continue
        dng_files = glob(os.path.join(folder_path, "**", "*.dng"), recursive=True)
        for dng_path in dng_files:
            base_name = os.path.splitext(os.path.basename(dng_path))[0]
            dng_dict[base_name] = (dng_path, score)
        print(f"  Score {score:02d}: {len(dng_files)} DNG files")
        total += len(dng_files)
    print(f"  Total DNG files: {total}")
    return dng_dict


def discover_tiff_files(base_path, score_range):
    tiff_dict = {}
    total = 0

    score_folders = glob(os.path.join(base_path, "Score=*"))
    if score_folders:
        print("  Found Score=XX subfolder structure")
        for sf in sorted(score_folders):
            score_str = os.path.basename(sf).split("=")[-1]
            score = int(score_str)
            if score not in score_range:
                continue
            tif_files = glob(os.path.join(sf, "*.tif")) + glob(os.path.join(sf, "*.tiff"))
            for tif_path in tif_files:
                base_name = os.path.splitext(os.path.basename(tif_path))[0]
                tiff_dict[base_name] = (tif_path, score)
            print(f"  Score {score:02d}: {len(tif_files)} TIFF files")
            total += len(tif_files)
    else:
        numbered_folders = [d for d in os.listdir(base_path)
                             if os.path.isdir(os.path.join(base_path, d)) and d.isdigit()]
        if numbered_folders:
            print("  Found numbered subfolder structure")
            for folder_name in sorted(numbered_folders):
                score = int(folder_name)
                if score not in score_range:
                    continue
                folder_path = os.path.join(base_path, folder_name)
                tif_files = glob(os.path.join(folder_path, "*.tif")) + glob(os.path.join(folder_path, "*.tiff"))
                for tif_path in tif_files:
                    base_name = os.path.splitext(os.path.basename(tif_path))[0]
                    tiff_dict[base_name] = (tif_path, score)
                print(f"  Score {score:02d}: {len(tif_files)} TIFF files")
                total += len(tif_files)
        else:
            print("  Scanning flat folder for TIFF files...")
            tif_files = glob(os.path.join(base_path, "**", "*.tif"), recursive=True) + \
                        glob(os.path.join(base_path, "**", "*.tiff"), recursive=True)
            for tif_path in tif_files:
                base_name = os.path.splitext(os.path.basename(tif_path))[0]
                parent = os.path.basename(os.path.dirname(tif_path))
                if parent.isdigit():
                    score = int(parent)
                elif 'Score=' in parent:
                    score = int(parent.split('=')[-1])
                else:
                    continue
                if score in score_range:
                    tiff_dict[base_name] = (tif_path, score)
                    total += 1

    print(f"  Total TIFF files: {total}")
    return tiff_dict


print("=" * 80)
print("DISCOVERING DNG FILES")
print("=" * 80)
dng_dict = discover_dng_files(DNG_BASE_PATH, DNG_FOLDER_PATTERN, SCORE_RANGE)

print("\n" + "=" * 80)
print("DISCOVERING TIFF FILES")
print("=" * 80)
tiff_dict = discover_tiff_files(TIFF_BASE_PATH, SCORE_RANGE)


In [ ]:
# Match DNG to TIFF by base filename
matched_pairs = []
unmatched_dng = []
unmatched_tiff = []

for base_name, (dng_path, dng_score) in dng_dict.items():
    if base_name in tiff_dict:
        tiff_path, tiff_score = tiff_dict[base_name]
        matched_pairs.append({
            'base_name': base_name,
            'dng_path': dng_path,
            'tiff_path': tiff_path,
            'dng_score': dng_score,
            'tiff_score': tiff_score,
            'score_match': dng_score == tiff_score
        })
    else:
        unmatched_dng.append(base_name)

for base_name in tiff_dict:
    if base_name not in dng_dict:
        unmatched_tiff.append(base_name)

print(f"\nMatching Results:")
print(f"  Matched pairs: {len(matched_pairs)}")
print(f"  Unmatched DNG (no TIFF): {len(unmatched_dng)}")
print(f"  Unmatched TIFF (no DNG): {len(unmatched_tiff)}")

if matched_pairs:
    score_mismatches = [p for p in matched_pairs if not p['score_match']]
    print(f"  Score mismatches: {len(score_mismatches)}")
    if score_mismatches:
        print("  WARNING: Some pairs have different scores!")
        for p in score_mismatches[:5]:
            print(f"    {p['base_name']}: DNG={p['dng_score']}, TIFF={p['tiff_score']}")

if unmatched_dng:
    print(f"\n  First unmatched DNG filenames:")
    for name in unmatched_dng[:5]:
        print(f"    {name}")
if unmatched_tiff:
    print(f"\n  First unmatched TIFF filenames:")
    for name in unmatched_tiff[:5]:
        print(f"    {name}")

# Deterministic, guaranteed row order (sorted by base_name) before any K-fold split is computed.
# This matters: StratifiedKFold/RepeatedStratifiedKFold(shuffle=True, random_state=SEED) produce
# DIFFERENT fold membership for the identical seed if the input row order differs, since shuffling
# operates on positional indices. matched_pairs was built by iterating dng_dict.items() (grouped by
# score folder, not sorted) -- fixing the order here makes this notebook's fold membership
# reproducible run-to-run and comparable against any other notebook that also sorts by base_name.
matched_pairs = sorted(matched_pairs, key=lambda p: p['base_name'])

# NEW: dataframe view of the matched pairs, used from section 7 onward for the paired,
# cross-validated CNN training (the fold assignment is computed once on this table and
# reused identically for both sources).
matched_df = pd.DataFrame(matched_pairs)
matched_df["score"] = matched_df["dng_score"].astype(float)
print(f"\nClass distribution (matched pairs):")
print(matched_df["score"].value_counts().sort_index())


## 3. Visual Comparison: DNG vs CC TIFF

Load matched pairs and display side-by-side for visual inspection. **Unchanged from the original notebook.**

In [ ]:
def load_dng_as_float(dng_path):
    """
    Load DNG file using rawpy and convert to float [0, 1].
    Uses camera white balance, XYZ output, converted to ACES2065-1.
    """
    with rawpy.imread(dng_path) as raw:
        rgb = raw.postprocess(
            gamma=(1, 1),
            use_camera_wb=True,
            use_auto_wb=False,
            no_auto_bright=True,
            output_color=rawpy.ColorSpace.XYZ,
            output_bps=16
        )
    img_float = rgb.astype(np.float32) / 65535.0
    img_aces = colour.XYZ_to_RGB(img_float, colourspace='ACES2065-1')
    return np.clip(img_aces, 0, 1)


def load_tiff_as_float(tiff_path):
    """
    Load 16-bit TIFF and normalize to float [0, 1]. Fixed /65535 scale —
    no per-image contrast stretching.
    """
    img = io.imread(tiff_path)
    if img.dtype == np.uint16:
        return img.astype(np.float32) / 65535.0
    elif img.dtype == np.uint8:
        return img.astype(np.float32) / 255.0
    else:
        return img.astype(np.float32)


def display_to_srgb(img_aces):
    """Convert ACES2065-1 float image to sRGB for display."""
    xyz = colour.RGB_to_XYZ(img_aces, colourspace='ACES2065-1')
    srgb = colour.XYZ_to_RGB(xyz, colourspace='sRGB')
    return np.clip(srgb, 0, 1)


# Select a subset of matched pairs for visual comparison (one per score)
if matched_pairs:
    pairs_by_score = {}
    for p in matched_pairs:
        s = p['dng_score']
        if s not in pairs_by_score:
            pairs_by_score[s] = p

    n_samples = min(9, len(pairs_by_score))
    sample_pairs = [pairs_by_score[s] for s in sorted(pairs_by_score.keys())[:n_samples]]

    print(f"Loading {n_samples} sample pairs for visual comparison...")

    fig, axes = plt.subplots(n_samples, 3, figsize=(18, 4 * n_samples))
    if n_samples == 1:
        axes = axes[np.newaxis, :]

    for i, pair in enumerate(tqdm(sample_pairs, desc="Loading pairs")):
        dng_img = load_dng_as_float(pair['dng_path'])
        tiff_img = load_tiff_as_float(pair['tiff_path'])

        dng_display = display_to_srgb(dng_img)
        tiff_display_aces = display_to_srgb(tiff_img)

        if dng_img.shape != tiff_img.shape:
            tiff_resized = resize(tiff_img, dng_img.shape[:2], anti_aliasing=True)
        else:
            tiff_resized = tiff_img

        diff = np.abs(dng_img - tiff_resized)

        axes[i, 0].imshow(dng_display)
        axes[i, 0].set_title(f"DNG (Score={pair['dng_score']})", fontsize=12, fontweight='bold')
        axes[i, 0].axis('off')

        axes[i, 1].imshow(tiff_display_aces)
        axes[i, 1].set_title(f"CC TIFF (Score={pair['tiff_score']})", fontsize=12, fontweight='bold')
        axes[i, 1].axis('off')

        axes[i, 2].imshow(diff * 5, cmap='hot')
        axes[i, 2].set_title(f"|DNG - TIFF| x5 (mean={diff.mean():.4f})", fontsize=12, fontweight='bold')
        axes[i, 2].axis('off')

    plt.suptitle('DNG vs CC TIFF Comparison (ACES2065-1 → sRGB display)', fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("No matched pairs found! Check paths and naming conventions.")


## 4. Statistical Comparison

**Unchanged from the original notebook** — same ΔE (CIEDE2000) methodology, same loaders as training now uses.

In [ ]:
N_STAT_SAMPLES = min(50, len(matched_pairs))
stat_pairs = matched_pairs[:N_STAT_SAMPLES]
stats_results = []

print(f"Computing statistics on {N_STAT_SAMPLES} matched pairs...")

for pair in tqdm(stat_pairs, desc="Computing stats"):
    dng_img = load_dng_as_float(pair['dng_path'])
    tiff_img = load_tiff_as_float(pair['tiff_path'])

    if dng_img.shape != tiff_img.shape:
        tiff_img = resize(tiff_img, dng_img.shape[:2], anti_aliasing=True)

    diff = dng_img - tiff_img
    abs_diff = np.abs(diff)

    channel_names = ['R/X', 'G/Y', 'B/Z']
    row = {
        'base_name': pair['base_name'], 'score': pair['dng_score'],
        'mean_abs_diff': abs_diff.mean(), 'max_abs_diff': abs_diff.max(), 'std_diff': diff.std(),
        'dng_mean': dng_img.mean(), 'tiff_mean': tiff_img.mean(),
        'dng_std': dng_img.std(), 'tiff_std': tiff_img.std(),
    }
    for c, name in enumerate(channel_names):
        row[f'{name}_mean_diff'] = diff[:, :, c].mean()
        row[f'{name}_std_diff'] = diff[:, :, c].std()

    h, w = dng_img.shape[:2]
    ch, cw = h // 4, w // 4
    dng_center = dng_img[ch:3 * ch, cw:3 * cw]
    tiff_center = tiff_img[ch:3 * ch, cw:3 * cw]
    dng_xyz = colour.RGB_to_XYZ(dng_center, colourspace='ACES2065-1')
    tiff_xyz = colour.RGB_to_XYZ(tiff_center, colourspace='ACES2065-1')
    dng_lab = colour.XYZ_to_Lab(dng_xyz)
    tiff_lab = colour.XYZ_to_Lab(tiff_xyz)
    deltaE = colour.difference.delta_E_CIE2000(dng_lab, tiff_lab)
    row['deltaE_mean'] = deltaE.mean()
    row['deltaE_max'] = deltaE.max()
    row['deltaE_std'] = deltaE.std()

    stats_results.append(row)

stats_df = pd.DataFrame(stats_results)
print("\n" + "=" * 80)
print("STATISTICAL SUMMARY: DNG vs CC TIFF")
print("=" * 80)
print(stats_df[['score', 'mean_abs_diff', 'max_abs_diff', 'std_diff',
                'deltaE_mean', 'deltaE_max', 'dng_mean', 'tiff_mean']].describe())

if stats_df['deltaE_mean'].mean() > 5:
    print(f"\n⚠️ HIGH COLOR DIFFERENCE DETECTED: mean ΔE={stats_df['deltaE_mean'].mean():.2f} (CC pipeline may have issues!)")


## 5. CNN Model Definition

**Unchanged** — same `RegressionCNN` architecture.

In [ ]:
class RegressionCNN(nn.Module):
    """Custom 2D CNN for regression."""
    def __init__(self, input_channels=3, img_height=64, img_width=64, depth=16):
        super(RegressionCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(input_channels, depth, kernel_size=3, padding=1), nn.BatchNorm2d(depth), nn.ReLU(),
            nn.Conv2d(depth, depth, kernel_size=3, padding=1), nn.BatchNorm2d(depth), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(depth, depth * 2, kernel_size=3, padding=1), nn.BatchNorm2d(depth * 2), nn.ReLU(),
            nn.Conv2d(depth * 2, depth * 2, kernel_size=3, padding=1), nn.BatchNorm2d(depth * 2), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(depth * 2, depth * 4, kernel_size=3, padding=1), nn.BatchNorm2d(depth * 4), nn.ReLU(),
            nn.Conv2d(depth * 4, depth * 4, kernel_size=3, padding=1), nn.BatchNorm2d(depth * 4), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(depth * 4, depth * 8, kernel_size=3, padding=1), nn.BatchNorm2d(depth * 8), nn.ReLU(),
            nn.Conv2d(depth * 8, depth * 8, kernel_size=3, padding=1), nn.BatchNorm2d(depth * 8), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(depth * 8, depth * 16, kernel_size=3, padding=1), nn.BatchNorm2d(depth * 16), nn.ReLU(),
            nn.Conv2d(depth * 16, depth * 16, kernel_size=3, padding=1), nn.BatchNorm2d(depth * 16), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(depth * 16, depth * 32, kernel_size=3, padding=1), nn.BatchNorm2d(depth * 32), nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.flattened_size = depth * img_height * img_width // 128
        self.fc_layers = nn.Sequential(
            nn.Linear(self.flattened_size, self.flattened_size // 2), nn.ReLU(),
            nn.Linear(self.flattened_size // 2, self.flattened_size // 4), nn.ReLU(),
            nn.Linear(self.flattened_size // 4, 1),
            nn.ELU(),
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x

_m = RegressionCNN(3, IMG_SIZE[0], IMG_SIZE[1], DEPTH).to(device)
total_params = sum(p.numel() for p in _m.parameters())
print(f"Model OK | flattened_size={_m.flattened_size} | params={total_params:,}")
del _m


## 6. Custom Datasets & Transforms

**Unchanged** — same `TIFFRegressionDataset` / `DNGRegressionDataset` classes and augmentation pipeline.

In [ ]:
class TIFFRegressionDataset(Dataset):
    """Dataset for 16-bit TIFF images (CC pipeline output)."""
    def __init__(self, data_list, transform=None, target_noise_std=0.0):
        self.data_list = data_list
        self.transform = transform
        self.target_noise_std = target_noise_std

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        img_path, target = self.data_list[idx]
        img_array = io.imread(img_path)
        if img_array.dtype == np.uint16:
            img_array = img_array.astype(np.float32) / 65535.0
        elif img_array.dtype == np.uint8:
            img_array = img_array.astype(np.float32) / 255.0
        else:
            img_array = img_array.astype(np.float32)

        if img_array.ndim == 2:
            img_array = np.stack([img_array] * 3, axis=-1)
        elif img_array.shape[2] == 4:
            img_array = img_array[:, :, :3]

        image = torch.from_numpy(img_array.transpose(2, 0, 1)).float()
        if self.transform:
            image = self.transform(image)

        target_tensor = torch.tensor([target], dtype=torch.float32)
        if self.target_noise_std > 0:
            target_tensor = target_tensor + torch.randn(1) * self.target_noise_std
        return image, target_tensor


class DNGRegressionDataset(Dataset):
    """Dataset for DNG raw images (loaded with rawpy -> ACES2065-1)."""
    def __init__(self, data_list, transform=None, target_noise_std=0.0):
        self.data_list = data_list
        self.transform = transform
        self.target_noise_std = target_noise_std

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        img_path, target = self.data_list[idx]
        img_array = load_dng_as_float(img_path)
        img_array = np.clip(img_array, 0, 1).astype(np.float32)
        image = torch.from_numpy(img_array.transpose(2, 0, 1)).float()

        if self.transform:
            image = self.transform(image)

        target_tensor = torch.tensor([target], dtype=torch.float32)
        if self.target_noise_std > 0:
            target_tensor = target_tensor + torch.randn(1) * self.target_noise_std
        return image, target_tensor


train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomChoice([
        transforms.Lambda(lambda x: torch.rot90(x, k=0, dims=[1, 2])),
        transforms.Lambda(lambda x: torch.rot90(x, k=1, dims=[1, 2])),
        transforms.Lambda(lambda x: torch.rot90(x, k=2, dims=[1, 2])),
        transforms.Lambda(lambda x: torch.rot90(x, k=3, dims=[1, 2]))
    ]),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

val_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

print("Dataset classes and transforms defined.")


In [ ]:
def train_model(train_loader, val_loader, model_name="model", num_epochs=NUM_EPOCHS):
    """Train a CNN regression model and return history + best model. Unchanged from the
    original notebook — the fix is in HOW this gets called (section 9): with a controlled
    seed reset immediately before each call, and on paired, fold-specific data."""
    model = RegressionCNN(3, IMG_SIZE[0], IMG_SIZE[1], DEPTH).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=0.0)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=LR_FACTOR, patience=LR_PATIENCE
    )

    history = {'train_loss': [], 'val_loss': [], 'val_mae': [], 'val_rmse': [], 'val_r2': [], 'lr': []}
    best_val_loss = float('inf')
    best_model_state = None
    epochs_no_improve = 0
    best_epoch = 0

    print(f"\nTraining {model_name}...")
    print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>10} | {'Val MAE':>8} | {'Val RMSE':>9} | {'Val R²':>8} | {'LR':>10}")
    print("-" * 80)

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * images.size(0)
        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        all_preds, all_targets = [], []
        with torch.no_grad():
            for images, targets in val_loader:
                images, targets = images.to(device), targets.to(device)
                outputs = model(images)
                loss = criterion(outputs, targets)
                val_loss += loss.item() * images.size(0)
                all_preds.extend(outputs.cpu().numpy().flatten())
                all_targets.extend(targets.cpu().numpy().flatten())
        val_loss /= len(val_loader.dataset)

        all_preds = np.array(all_preds)
        all_targets = np.array(all_targets)
        val_mae = mean_absolute_error(all_targets, all_preds)
        val_rmse = np.sqrt(mean_squared_error(all_targets, all_preds))
        val_r2 = r2_score(all_targets, all_preds) if len(set(all_targets)) > 1 else float('nan')

        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_loss)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_mae'].append(val_mae)
        history['val_rmse'].append(val_rmse)
        history['val_r2'].append(val_r2)
        history['lr'].append(current_lr)

        if (epoch + 1) % 20 == 0 or epoch == 0:
            print(f"{epoch+1:>6} | {train_loss:>10.4f} | {val_loss:>10.4f} | {val_mae:>8.4f} | {val_rmse:>9.4f} | {val_r2:>8.4f} | {current_lr:>10.6f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            best_epoch = epoch + 1
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
            print(f"\n  Early stopping at epoch {epoch+1} (no improvement for {EARLY_STOPPING_PATIENCE} epochs)")
            break

    model.load_state_dict(best_model_state)
    print(f"\n  Best val loss: {best_val_loss:.4f} @ epoch {best_epoch} (loaded best weights)")
    history['best_epoch'] = best_epoch
    history['epochs_run'] = epoch + 1
    return model, history


def evaluate_model(model, val_loader):
    """Evaluate model and return predictions, targets, images, and metrics. Unchanged."""
    model.eval()
    all_preds, all_targets, all_images = [], [], []
    with torch.no_grad():
        for images, targets in val_loader:
            images = images.to(device)
            outputs = model(images)
            all_preds.extend(outputs.cpu().numpy().flatten())
            all_targets.extend(targets.cpu().numpy().flatten())
            all_images.extend(images.cpu().permute(0, 2, 3, 1).numpy())

    preds = np.array(all_preds)
    targets = np.array(all_targets)
    images = np.array(all_images)

    metrics = {
        'rmse': np.sqrt(mean_squared_error(targets, preds)),
        'mae': mean_absolute_error(targets, preds),
        'r2': r2_score(targets, preds) if len(set(targets)) > 1 else float('nan'),
        'acc_0': np.mean(np.round(preds) == np.round(targets)),
        'acc_1': np.mean(np.abs(np.round(preds) - np.round(targets)) <= 1),
    }
    return preds, targets, images, metrics

print("Training utilities defined.")


## 7. Prepare Paired Datasets for CNN Training

**This is the key fix.** Both `tiff_data_list` and `dng_data_list` are now built from
`matched_pairs` only — same length, same order, same underlying physical item at each position.
Previously, TIFF used all 213 discovered files and DNG used all 224 discovered files
independently, so the two models were not trained/evaluated on the same images.

In [ ]:
tiff_data_list = [(p['tiff_path'], float(p['dng_score'])) for p in matched_pairs]
dng_data_list = [(p['dng_path'], float(p['dng_score'])) for p in matched_pairs]

assert len(tiff_data_list) == len(dng_data_list) == len(matched_pairs)
print(f"Paired dataset size (both sources): {len(matched_pairs)} images")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data_list, title in [(axes[0], dng_data_list, 'DNG (matched)'), (axes[1], tiff_data_list, 'CC TIFF (matched)')]:
    targets = [t for _, t in data_list]
    unique, counts = np.unique(targets, return_counts=True)
    ax.bar(unique, counts, color='steelblue', edgecolor='black', linewidth=1.5)
    ax.set_xlabel('Score', fontweight='bold'); ax.set_ylabel('Count', fontweight='bold')
    ax.set_title(f'{title} Distribution (N={len(data_list)})', fontweight='bold')
    ax.set_xticks(unique)
    ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()


## 8. Repeated Stratified K-Fold Cross-Validation Setup

Computed **once**, on the matched-pairs score labels (already sorted by `base_name`), and reused
identically for both sources — split *i*'s validation indices are the same physical images whether
we're training the DNG model or the TIFF model. `K_FOLDS × N_REPEATS` = 15 independent splits
instead of 5, so the final comparison is far less sensitive to any single split's luck.

In [ ]:
rskf = RepeatedStratifiedKFold(n_splits=K_FOLDS, n_repeats=N_REPEATS, random_state=SEED)
fold_splits = list(rskf.split(matched_df, matched_df["score"]))
N_SPLITS_TOTAL = len(fold_splits)
assert N_SPLITS_TOTAL == K_FOLDS * N_REPEATS

print(f"Created {N_SPLITS_TOTAL} splits ({N_REPEATS} repeats x {K_FOLDS} folds) from {len(matched_df)} matched pairs.")
for i, (tr_idx, va_idx) in enumerate(fold_splits):
    repeat_i, fold_i = divmod(i, K_FOLDS)
    print(f"  Repeat {repeat_i+1}, Fold {fold_i+1}: train={len(tr_idx)}  val={len(va_idx)}  "
          f"val class counts={dict(matched_df.iloc[va_idx]['score'].value_counts().sort_index())}")


## 9. Cross-Validated Training (paired, controlled seeding)

For each split: `reset_seed(SEED + split_idx)` is called **identically** right before building the
TIFF model and again right before building the DNG model, so within a split both sources start from
the same initial weights and see the same augmentation RNG stream — the only thing that differs
between the two runs in a split is the image source itself. Because `N_REPEATS > 1`, every image is
held out `N_REPEATS` times across different fold partitions; its final OOF prediction is the
average of those `N_REPEATS` independent held-out predictions.

⚠️ This trains `K_FOLDS × N_REPEATS × 2 = 30` models, each up to 400 epochs — 3x the v2
workload. Expect a long runtime even on GPU. Reduce `N_REPEATS` (to 1 for the old behavior),
`K_FOLDS`, or `NUM_EPOCHS` above for a quicker run.

In [ ]:
n = len(matched_pairs)
tiff_oof_preds_by_repeat = np.full((N_REPEATS, n), np.nan)
tiff_oof_trues_by_repeat = np.full((N_REPEATS, n), np.nan)
dng_oof_preds_by_repeat = np.full((N_REPEATS, n), np.nan)
dng_oof_trues_by_repeat = np.full((N_REPEATS, n), np.nan)
tiff_oof_images = np.zeros((n, IMG_SIZE[0], IMG_SIZE[1], 3), dtype=np.float32)
dng_oof_images = np.zeros((n, IMG_SIZE[0], IMG_SIZE[1], 3), dtype=np.float32)

tiff_split_metrics, dng_split_metrics = [], []
tiff_split_histories, dng_split_histories = [], []

for split_idx, (train_idx, val_idx) in enumerate(fold_splits):
    repeat_idx, fold_idx = divmod(split_idx, K_FOLDS)
    split_seed = SEED + split_idx
    print(f"\n{'#'*90}\nREPEAT {repeat_idx+1}/{N_REPEATS}, FOLD {fold_idx+1}/{K_FOLDS}  (seed={split_seed})\n{'#'*90}")

    tiff_train = [tiff_data_list[i] for i in train_idx]
    tiff_val = [tiff_data_list[i] for i in val_idx]
    dng_train = [dng_data_list[i] for i in train_idx]
    dng_val = [dng_data_list[i] for i in val_idx]

    tiff_train_ds = TIFFRegressionDataset(tiff_train, transform=train_transform, target_noise_std=0.15)
    tiff_val_ds = TIFFRegressionDataset(tiff_val, transform=val_transform)
    tiff_train_loader = DataLoader(tiff_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    tiff_val_loader = DataLoader(tiff_val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    dng_train_ds = DNGRegressionDataset(dng_train, transform=train_transform, target_noise_std=0.15)
    dng_val_ds = DNGRegressionDataset(dng_val, transform=val_transform)
    dng_train_loader = DataLoader(dng_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    dng_val_loader = DataLoader(dng_val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    # --- TIFF: controlled seed reset right before model construction ---
    reset_seed(split_seed)
    tiff_model, tiff_history = train_model(tiff_train_loader, tiff_val_loader,
                                            model_name=f"TIFF (CC Pipeline) - repeat {repeat_idx+1} fold {fold_idx+1}")
    tiff_preds, tiff_targets_eval, tiff_images, tiff_metrics = evaluate_model(tiff_model, tiff_val_loader)
    tiff_metrics.update({'repeat': repeat_idx + 1, 'fold': fold_idx + 1, 'split': split_idx + 1,
                          'best_epoch': tiff_history['best_epoch'], 'epochs_run': tiff_history['epochs_run']})
    tiff_split_metrics.append(tiff_metrics)
    tiff_split_histories.append(tiff_history)
    tiff_oof_preds_by_repeat[repeat_idx, val_idx] = tiff_preds
    tiff_oof_trues_by_repeat[repeat_idx, val_idx] = tiff_targets_eval
    tiff_oof_images[val_idx] = tiff_images

    # --- DNG: SAME seed as TIFF for this split, reset right before model construction ---
    reset_seed(split_seed)
    dng_model, dng_history = train_model(dng_train_loader, dng_val_loader,
                                          model_name=f"DNG (Raw Source) - repeat {repeat_idx+1} fold {fold_idx+1}")
    dng_preds, dng_targets_eval, dng_images, dng_metrics = evaluate_model(dng_model, dng_val_loader)
    dng_metrics.update({'repeat': repeat_idx + 1, 'fold': fold_idx + 1, 'split': split_idx + 1,
                         'best_epoch': dng_history['best_epoch'], 'epochs_run': dng_history['epochs_run']})
    dng_split_metrics.append(dng_metrics)
    dng_split_histories.append(dng_history)
    dng_oof_preds_by_repeat[repeat_idx, val_idx] = dng_preds
    dng_oof_trues_by_repeat[repeat_idx, val_idx] = dng_targets_eval
    dng_oof_images[val_idx] = dng_images

    print(f"\nRepeat {repeat_idx+1}, Fold {fold_idx+1} summary: TIFF RMSE={tiff_metrics['rmse']:.4f} R2={tiff_metrics['r2']:.4f}  |  "
          f"DNG RMSE={dng_metrics['rmse']:.4f} R2={dng_metrics['r2']:.4f}")

assert not np.isnan(tiff_oof_preds_by_repeat).any() and not np.isnan(dng_oof_preds_by_repeat).any(), \
    "Every matched pair should receive exactly one OOF prediction per repeat, per source"

# Repeat-averaged OOF: each image was held out once per repeat (in a different fold each time);
# average its N_REPEATS independent held-out predictions into a single, lower-variance estimate.
tiff_oof_preds = np.nanmean(tiff_oof_preds_by_repeat, axis=0)
tiff_oof_trues = matched_df["score"].values.astype(float)
dng_oof_preds = np.nanmean(dng_oof_preds_by_repeat, axis=0)
dng_oof_trues = matched_df["score"].values.astype(float)

print("\nCross-validated training complete. Each matched pair now has N_REPEATS out-of-fold")
print("predictions per source, averaged into one repeat-averaged OOF prediction per source.")


## 10. Per-Split Results (mean ± std across all `K_FOLDS × N_REPEATS` splits)

In [ ]:
def split_metrics_table(split_metrics, tag):
    df = pd.DataFrame(split_metrics)
    df.insert(0, 'source', tag)
    return df

tiff_split_df = split_metrics_table(tiff_split_metrics, 'TIFF')
dng_split_df = split_metrics_table(dng_split_metrics, 'DNG')
all_splits_df = pd.concat([dng_split_df, tiff_split_df], ignore_index=True)
display(all_splits_df[['source', 'repeat', 'fold', 'rmse', 'mae', 'r2', 'acc_0', 'acc_1', 'best_epoch', 'epochs_run']])

print(f"\nMean ± std across all {len(dng_split_df)} splits per source:")
for tag, df in [('DNG', dng_split_df), ('TIFF', tiff_split_df)]:
    print(f"\n{tag}:")
    for metric in ['rmse', 'mae', 'r2', 'acc_0', 'acc_1']:
        print(f"  {metric:8s}: {df[metric].mean():.4f} ± {df[metric].std():.4f}")


## 10b. Paired Significance Test (DNG − TIFF, per split)

Because DNG and TIFF always share identical split membership (same `train_idx`/`val_idx` in every
split), the per-split `(DNG − TIFF)` differences are naturally **paired** — far more powerful
than treating the two sources as independent samples. A paired t-test and a Wilcoxon signed-rank
test (distribution-free, more robust with small samples) both test whether that paired difference
is distinguishable from zero.

In [ ]:
def paired_test(dng_df, tiff_df, metric):
    dng_vals = dng_df.sort_values('split')[metric].values
    tiff_vals = tiff_df.sort_values('split')[metric].values
    diffs = dng_vals - tiff_vals
    t_stat, t_p = ttest_rel(dng_vals, tiff_vals)
    try:
        w_stat, w_p = wilcoxon(diffs)
    except ValueError:
        w_stat, w_p = float('nan'), float('nan')  # all-zero differences
    return {
        'metric': metric, 'mean_diff (DNG-TIFF)': diffs.mean(), 'std_diff': diffs.std(),
        't_stat': t_stat, 't_pvalue': t_p, 'wilcoxon_stat': w_stat, 'wilcoxon_pvalue': w_p,
    }

paired_results = [paired_test(dng_split_df, tiff_split_df, m) for m in ['rmse', 'mae', 'r2', 'acc_0', 'acc_1']]
paired_results_df = pd.DataFrame(paired_results)
print(f"Paired tests across all {len(dng_split_df)} splits (alpha=0.05):")
display(paired_results_df)

for row in paired_results:
    sig = 'SIGNIFICANT' if row['t_pvalue'] < 0.05 else 'not significant'
    print(f"  {row['metric']:6s}: mean(DNG-TIFF)={row['mean_diff (DNG-TIFF)']:+.4f}  "
          f"paired t-test p={row['t_pvalue']:.4f} ({sig})  wilcoxon p={row['wilcoxon_pvalue']:.4f}")


## 11. Pooled, Repeat-Averaged Out-of-Fold Metrics Comparison

The headline, leak-free result: every matched image is held out `N_REPEATS` times (once per
repeat), and its final prediction is the **average** of those `N_REPEATS` independent held-out
predictions — lower variance than a single OOF pass, still fully leak-free, for both sources on
the identical set of images.

In [ ]:
def compute_metrics(preds, targets):
    return {
        'rmse': np.sqrt(mean_squared_error(targets, preds)),
        'mae': mean_absolute_error(targets, preds),
        'r2': r2_score(targets, preds),
        'acc_0': np.mean(np.round(preds) == np.round(targets)),
        'acc_1': np.mean(np.abs(np.round(preds) - np.round(targets)) <= 1),
    }

tiff_metrics = compute_metrics(tiff_oof_preds, tiff_oof_trues)
dng_metrics = compute_metrics(dng_oof_preds, dng_oof_trues)

print("\n" + "=" * 80)
print(f"POOLED, REPEAT-AVERAGED OOF METRICS COMPARISON  (n={n} matched pairs, {K_FOLDS}-fold x {N_REPEATS}-repeat CV)")
print("=" * 80)
print(f"{'Metric':<20} | {'DNG (Raw)':>12} | {'TIFF (CC)':>12} | {'Winner':>10}")
print("-" * 60)

metrics_compare = [
    ('RMSE ↓', 'rmse', False), ('MAE ↓', 'mae', False), ('R² ↑', 'r2', True),
    ('Accuracy@0 ↑', 'acc_0', True), ('Accuracy@1 ↑', 'acc_1', True),
]
for name, key, higher_better in metrics_compare:
    dv, tv = dng_metrics[key], tiff_metrics[key]
    winner = ('DNG' if dv > tv else 'TIFF') if higher_better else ('DNG' if dv < tv else 'TIFF')
    print(f"{name:<20} | {dv:>12.4f} | {tv:>12.4f} | {winner:>10}")
print("=" * 80)
print("Note: 'Winner' above is just the numerically larger/smaller value — it does NOT mean the")
print("difference is statistically significant. Check section 10b's paired test before concluding")
print("either source is actually better.")

# tiff_preds/tiff_targets_eval/tiff_images (and dng_*) are reused by name in the sections below,
# now pointing at the pooled OOF arrays instead of a single held-out split.
tiff_preds, tiff_targets_eval, tiff_images = tiff_oof_preds, tiff_oof_trues, tiff_oof_images
dng_preds, dng_targets_eval, dng_images = dng_oof_preds, dng_oof_trues, dng_oof_images


## 12. Per-Class Statistical Summary (OOF)

In [ ]:
def per_class_summary(preds, targets, dataset_name):
    print(f"\n{'=' * 80}")
    print(f"PER-CLASS SUMMARY (OOF): {dataset_name}")
    print(f"{'=' * 80}")
    print(f"{'Score':<7} | {'N':>4} | {'Mean Pred':>10} | {'Std Pred':>9} | {'MAE':>7} | {'RMSE':>7} | {'Acc@0':>6} | {'Acc@1':>6}")
    print("-" * 75)

    results = []
    for score in sorted(np.unique(targets)):
        mask = targets == score
        p = preds[mask]
        n_c = len(p)
        mean_pred, std_pred = p.mean(), p.std()
        mae = np.abs(p - score).mean()
        rmse = np.sqrt(((p - score) ** 2).mean())
        acc0 = np.mean(np.round(p) == score)
        acc1 = np.mean(np.abs(np.round(p) - score) <= 1)
        print(f"{int(score):<7} | {n_c:>4} | {mean_pred:>10.3f} | {std_pred:>9.3f} | {mae:>7.3f} | {rmse:>7.3f} | {acc0:>6.1%} | {acc1:>6.1%}")
        results.append({'score': int(score), 'n': n_c, 'mean_pred': mean_pred, 'std_pred': std_pred,
                         'mae': mae, 'rmse': rmse, 'acc0': acc0, 'acc1': acc1})
    return pd.DataFrame(results)

tiff_class_df = per_class_summary(tiff_preds, tiff_targets_eval, "TIFF (CC Pipeline)")
dng_class_df = per_class_summary(dng_preds, dng_targets_eval, "DNG (Raw Source)")


## 13. Confusion Matrix (OOF)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, preds, targets, title in [
    (axes[0], dng_preds, dng_targets_eval, 'DNG (Raw Source)'),
    (axes[1], tiff_preds, tiff_targets_eval, 'TIFF (CC Pipeline)')
]:
    preds_rounded = np.clip(np.round(preds), matched_df['score'].min(), matched_df['score'].max()).astype(int)
    targets_int = targets.astype(int)
    labels = sorted(np.unique(np.concatenate([targets_int, preds_rounded])))
    cm = confusion_matrix(targets_int, preds_rounded, labels=labels)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=cm, fmt='d', cmap='Blues', ax=ax,
                xticklabels=labels, yticklabels=labels, cbar_kws={'label': 'Proportion'})
    ax.set_xlabel('Predicted Score', fontweight='bold'); ax.set_ylabel('True Score', fontweight='bold')
    ax.set_title(f'Confusion Matrix (OOF): {title}', fontweight='bold')

plt.suptitle('Confusion Matrices — pooled out-of-fold predictions (values=counts, color=row-normalized)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 14. Predicted vs True Scatter Plots (OOF)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, preds, targets, metrics, title in [
    (axes[0], dng_preds, dng_targets_eval, dng_metrics, 'DNG (Raw Source)'),
    (axes[1], tiff_preds, tiff_targets_eval, tiff_metrics, 'TIFF (CC Pipeline)')
]:
    ax.scatter(targets, preds, alpha=0.5, s=40, edgecolors='black', linewidth=0.3)
    ax.plot([0, 10], [0, 10], 'r--', linewidth=2, label='Perfect')
    ax.set_xlabel('True Score', fontweight='bold'); ax.set_ylabel('Predicted Score', fontweight='bold')
    ax.set_title(f"{title}\nR²={metrics['r2']:.3f} | RMSE={metrics['rmse']:.3f} | Acc@1={metrics['acc_1']:.1%}", fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 10); ax.set_ylim(0, 10)

plt.suptitle('Predicted vs True Score (pooled OOF)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


## 15. Frieze of Predictions (Per Score, Max 15 per Line, OOF)

In [ ]:
def plot_frieze_per_score(preds, targets, images, dataset_name, max_per_line=15):
    unique_scores = sorted(np.unique(targets))
    n_scores = len(unique_scores)

    fig, axes = plt.subplots(n_scores, max_per_line, figsize=(max_per_line * 1.2, n_scores * 1.8))
    if n_scores == 1:
        axes = axes[np.newaxis, :]

    for row, score in enumerate(unique_scores):
        mask = targets == score
        score_preds = preds[mask]
        score_images = images[mask]

        sort_idx = np.argsort(score_preds)
        score_preds = score_preds[sort_idx]
        score_images = score_images[sort_idx]

        n_show = min(len(score_preds), max_per_line)
        if len(score_preds) > max_per_line:
            indices = np.linspace(0, len(score_preds) - 1, max_per_line, dtype=int)
            score_preds = score_preds[indices]
            score_images = score_images[indices]
            n_show = max_per_line

        for col in range(max_per_line):
            ax = axes[row, col]
            if col < n_show:
                img = score_images[col] * 0.5 + 0.5  # undo Normalize(mean=0.5, std=0.5)
                img = np.clip(img, 0, 1)
                ax.imshow(img)
                pred_val = score_preds[col]
                error = abs(pred_val - score)
                color = 'green' if error < 1.0 else ('orange' if error < 2.0 else 'red')
                ax.set_title(f'{pred_val:.1f}', fontsize=8, color=color, fontweight='bold')
            ax.axis('off')

        axes[row, 0].set_ylabel(f'Score {int(score)}', fontsize=11, fontweight='bold', rotation=0, labelpad=45, va='center')

    plt.suptitle(f'Frieze of Predictions (OOF): {dataset_name}\n(green: error<1, orange: error<2, red: error≥2)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

plot_frieze_per_score(tiff_preds, tiff_targets_eval, tiff_images, "TIFF (CC Pipeline)", max_per_line=15)


In [ ]:
plot_frieze_per_score(dng_preds, dng_targets_eval, dng_images, "DNG (Raw Source)", max_per_line=15)


## 16. Final Comparison Summary

In [ ]:
comparison_df = pd.DataFrame({
    'Metric': ['RMSE', 'MAE', 'R²', 'Accuracy@0 (exact)', 'Accuracy@1 (±1)'],
    'DNG (Raw Source)': [f"{dng_metrics['rmse']:.4f}", f"{dng_metrics['mae']:.4f}", f"{dng_metrics['r2']:.4f}",
                          f"{dng_metrics['acc_0']:.2%}", f"{dng_metrics['acc_1']:.2%}"],
    'TIFF (CC Pipeline)': [f"{tiff_metrics['rmse']:.4f}", f"{tiff_metrics['mae']:.4f}", f"{tiff_metrics['r2']:.4f}",
                           f"{tiff_metrics['acc_0']:.2%}", f"{tiff_metrics['acc_1']:.2%}"],
})

print("\n" + "=" * 80)
print(f"FINAL COMPARISON REPORT (pooled repeat-averaged OOF, {K_FOLDS}-fold x {N_REPEATS}-repeat CV, paired matched images, controlled seeding)")
print("=" * 80)
print(comparison_df.to_string(index=False))

print(f"\n\nTraining config:")
print(f"  Image size: {IMG_SIZE}  |  Batch size: {BATCH_SIZE}  |  Max epochs/split: {NUM_EPOCHS}")
print(f"  Early stopping: {EARLY_STOPPING_PATIENCE}  |  LR patience: {LR_PATIENCE}  |  LR factor: {LR_FACTOR}")
print(f"  Architecture depth: {DEPTH}  |  K-folds: {K_FOLDS}  |  Repeats: {N_REPEATS}  |  Matched pairs used: {n}")

print(f"\nEpochs run per split: DNG={[m['epochs_run'] for m in dng_split_metrics]}")
print(f"                       TIFF={[m['epochs_run'] for m in tiff_split_metrics]}")
print(f"Best epoch per split:  DNG={[m['best_epoch'] for m in dng_split_metrics]}")
print(f"                       TIFF={[m['best_epoch'] for m in tiff_split_metrics]}")

print(f"\n{'=' * 80}")
print("VERDICT A -- simple threshold (R2 difference > 0.05 on the pooled, repeat-averaged OOF)")
print(f"{'=' * 80}")
if dng_metrics['r2'] > tiff_metrics['r2'] + 0.05:
    print("DNG raw images give BETTER regression performance by this threshold.")
elif tiff_metrics['r2'] > dng_metrics['r2'] + 0.05:
    print("CC TIFF images give BETTER regression performance by this threshold.")
else:
    print("Both sources give SIMILAR regression performance within this comparison's noise level.")

r2_row = paired_results_df[paired_results_df['metric'] == 'r2'].iloc[0]
print(f"\n{'=' * 80}")
print("VERDICT B -- paired significance test across all splits (more rigorous than a flat threshold)")
print(f"{'=' * 80}")
print(f"Mean (DNG - TIFF) R2 = {r2_row['mean_diff (DNG-TIFF)']:+.4f}  |  paired t-test p={r2_row['t_pvalue']:.4f}  |  wilcoxon p={r2_row['wilcoxon_pvalue']:.4f}")
if r2_row['t_pvalue'] < 0.05:
    winner = 'DNG' if r2_row['mean_diff (DNG-TIFF)'] > 0 else 'TIFF'
    print(f"-> Statistically significant at alpha=0.05: {winner} performs better on R2 across splits.")
else:
    print("-> Not statistically significant at alpha=0.05: cannot reject 'DNG and TIFF perform the same'.")

print("\nCheck section 10's per-split std and section 10b's paired test alongside this verdict — with only "
      f"{n} matched images split across {K_FOLDS} folds, per-split sample sizes are small "
      "and single-run differences of a few points can reflect sampling noise.")


In [ ]:
# Side-by-side per-class comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

x = np.arange(len(tiff_class_df))
width = 0.35

ax = axes[0]
ax.bar(x - width/2, dng_class_df['acc0'] * 100, width, label='DNG', color='#FF6B6B', edgecolor='black')
ax.bar(x + width/2, tiff_class_df['acc0'] * 100, width, label='TIFF', color='#4D96FF', edgecolor='black')
ax.set_xlabel('Score', fontweight='bold'); ax.set_ylabel('Accuracy@0 (%)', fontweight='bold')
ax.set_title('Per-Class Exact Accuracy (OOF)', fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(tiff_class_df['score'].astype(int))
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

ax = axes[1]
ax.bar(x - width/2, dng_class_df['rmse'], width, label='DNG', color='#FF6B6B', edgecolor='black')
ax.bar(x + width/2, tiff_class_df['rmse'], width, label='TIFF', color='#4D96FF', edgecolor='black')
ax.set_xlabel('Score', fontweight='bold'); ax.set_ylabel('RMSE', fontweight='bold')
ax.set_title('Per-Class RMSE (OOF)', fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(tiff_class_df['score'].astype(int))
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Per-Class Performance: DNG vs TIFF (pooled OOF)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
